In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
netflix_dataset=pd.read_csv('/content/drive/MyDrive/Netflix/Copy of combined_data_1.txt.zip',header=None,names=['Cust_Id','Ratings'],usecols=[0,1])

In [ ]:
netflix_dataset

In [ ]:
# we have to find movies count and customers count and ratings count

In [ ]:
netflix_dataset.isnull().sum()

In [ ]:
movies_count=netflix_dataset.isnull().sum()
movies_count=movies_count['Ratings']
movies_count

In [ ]:
total_count=netflix_dataset['Cust_Id'].nunique()
total_count

In [ ]:
customer_count=total_count-movies_count
customer_count

In [ ]:
rating_count=netflix_dataset['Cust_Id'].count()-movies_count
rating_count

In [ ]:
stars=netflix_dataset['Ratings'].value_counts()
stars

In [ ]:
movie_id=None
movie_np=[]

for customer in netflix_dataset['Cust_Id']:
  if ":" in customer:
    movie_id=int(customer.replace(":",""))   #1: --> 1

  movie_np.append(movie_id)

In [ ]:
netflix_dataset['Movie_Id']=movie_np
netflix_dataset

In [ ]:
netflix_dataset.dropna(inplace=True)
netflix_dataset

In [ ]:
netflix_dataset.info()

In [ ]:
netflix_dataset['Cust_Id']=netflix_dataset['Cust_Id'].astype('int')

In [ ]:
netflix_dataset.info()

In [ ]:
movie_re_count=netflix_dataset['Movie_Id'].value_counts()
movie_re_count

In [ ]:
movie_re_count=netflix_dataset.groupby('Movie_Id')['Ratings'].count()
movie_re_count

In [ ]:
#now we are removing movies with less reviews because it is deviating us from building correct model

In [ ]:
bench_mark=round(movie_re_count.quantile(0.6),0)
bench_mark

In [ ]:
drop_movie_index=movie_re_count[movie_re_count<bench_mark].index
drop_movie_index

In [ ]:
#here we are removing the reviews given by the customers who have reviewed less number of movies

In [ ]:
cust_rev_count=netflix_dataset['Cust_Id'].value_counts()
cust_rev_count

In [ ]:
bench_mark_cus=round(cust_rev_count.quantile(0.6),0)
bench_mark_cus

In [ ]:
drop_cust_index=cust_rev_count[cust_rev_count<bench_mark_cus].index
drop_cust_index

In [ ]:
netflix_dataset=netflix_dataset[~netflix_dataset['Movie_Id'].isin(drop_movie_index)]
netflix_dataset=netflix_dataset[~netflix_dataset['Cust_Id'].isin(drop_cust_index)]
#removing from dataset

In [ ]:
netflix_dataset

In [ ]:
movie_title=pd.read_csv('/content/drive/MyDrive/Netflix/Copy of movie_titles.csv',encoding='ISO-8859-1',header=None,names=['Movie_Id','Year','Name'],usecols=[0,1,2])

In [ ]:
movie_title

In [ ]:
!pip install numpy==1.26.4

In [ ]:
!pip install scikit-surprise

In [ ]:
from surprise import SVD,Reader,Dataset
from surprise.model_selection import cross_validate

In [ ]:
reader=Reader()

In [ ]:
data=Dataset.load_from_df(netflix_dataset[['Movie_Id','Cust_Id','Ratings']][:100000],reader)

In [ ]:
data

In [ ]:
model=SVD()

In [ ]:
cross_validate(model,data,measures=['RMSE'],cv=3)

{'test_rmse': array([1.01452536, 1.01772645, 1.02286066]),
 'fit_time': (1.8675994873046875, 1.6599247455596924, 1.8802306652069092),
 'test_time': (0.16855955123901367, 0.18697738647460938, 0.1994938850402832)}

In [ ]:
user_rating=netflix_dataset[netflix_dataset['Cust_Id']==2643029]
user_rating

,Cust_Id,Ratings,Movie_Id
211028,2643029,4.0,30
539366,2643029,4.0,156
815159,2643029,3.0,191
1062501,2643029,5.0,241
1453806,2643029,4.0,299
...,...,...,...
22869123,2643029,4.0,4306
23291445,2643029,5.0,4356
23829756,2643029,4.0,4454
24018591,2643029,4.0,4488


In [ ]:
user_2643029=movie_title.copy()
user_2643029

,Movie_Id,Year,Name
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
...,...,...,...
17765,17766,2002.0,Where the Wild Things Are and Other Maurice Se...
17766,17767,2004.0,Fidel Castro: American Experience
17767,17768,2000.0,Epoch
17768,17769,2003.0,The Company


In [ ]:
user_2643029=user_2643029[~user_2643029['Movie_Id'].isin(drop_movie_index)]
user_2643029

,Movie_Id,Year,Name
2,3,1997.0,Character
4,5,2004.0,The Rise and Fall of ECW
5,6,1997.0,Sick
7,8,2004.0,What the #$*! Do We Know!?
15,16,1996.0,Screamers
...,...,...,...
17765,17766,2002.0,Where the Wild Things Are and Other Maurice Se...
17766,17767,2004.0,Fidel Castro: American Experience
17767,17768,2000.0,Epoch
17768,17769,2003.0,The Company


In [ ]:
user_2643029['Estimated']=user_2643029['Movie_Id'].apply(lambda x:model.predict(2643029,x).est)

/tmp/ipython-input-1051232510.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_2643029['Estimated']=user_2643029['Movie_Id'].apply(lambda x:model.predict(2643029,x).est)


In [ ]:
est=[]
for x in user_2643029['Movie_Id']:
  temp=model.predict(2643029,x).est
  est.append(temp)

user_2643029['Estimated']=est


/tmp/ipython-input-1770097549.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_2643029['Estimated']=est


In [ ]:
user_2643029

,Movie_Id,Year,Name,Estimated
2,3,1997.0,Character,3.586272
4,5,2004.0,The Rise and Fall of ECW,3.586272
5,6,1997.0,Sick,3.586272
7,8,2004.0,What the #$*! Do We Know!?,3.586272
15,16,1996.0,Screamers,3.586272
...,...,...,...,...
17765,17766,2002.0,Where the Wild Things Are and Other Maurice Se...,3.586272
17766,17767,2004.0,Fidel Castro: American Experience,3.586272
17767,17768,2000.0,Epoch,3.586272
17768,17769,2003.0,The Company,3.586272


In [ ]:
user_2643029=user_2643029.sort_values('Estimated',ascending=False)

In [ ]:
user_2643029.head()

,Movie_Id,Year,Name,Estimated
13874,13875,1982.0,Gilbert and Sullivan: The Mikado,3.884864
17450,17451,2000.0,Along for the Ride,3.870751
13510,13511,1993.0,Much Ado About Nothing,3.833598
11042,11043,1970.0,Mary Tyler Moore: Season 1,3.818100
10942,10943,2001.0,Ben Folds Five: Complete Sessions at West 54th,3.804085
